# Data Wrangling
Note: Some data wrangling processes such as merging datasets and engineering new features such as total_respiratory_admissions were conducted during milestone 1's data acquisition step. The code in the section focuses on data validation through handling duplicates, missing values, and producing the final analysis-ready dataset.

In [1]:
import _sqlite3
import pandas as pd
import numpy as np
import glob
from pathlib import Path
import shutil

# The project paths
ROOT = Path("..") # This notebook is not in the root
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"

db_path = PROCESSED_DIR / "milestone1.db"
conn = _sqlite3.connect(db_path)

df = pd.read_sql_query("SELECT * FROM merged_m1", conn)

### Data Validation: Duplicates

In [2]:
# Check for duplicates
print("Number of duplicates:", df.duplicated().sum())

Number of duplicates: 0


### Data Validation: Missing Values

In [3]:
# Calculation of value missing values and percentage rates(null values) for applicable variables
missing_values = df.isnull().sum()
missing_rates = df.isnull().mean() * 100

applicable_variables = [
    'covid_admissions', 
    'influenza_admissions', 
    'rsv_admissions', 
    'days_reported',
    'category',
    'defining_parameter'
]

print("Missingness Values (Total):")
print(missing_values[applicable_variables])

print("\nMissingness Rates (Percentage):")
print(missing_rates[applicable_variables])

Missingness Values (Total):
covid_admissions          55
influenza_admissions      55
rsv_admissions          8706
days_reported              0
category                   0
defining_parameter         0
dtype: int64

Missingness Rates (Percentage):
covid_admissions         0.420489
influenza_admissions     0.420489
rsv_admissions          66.559633
days_reported            0.000000
category                 0.000000
defining_parameter       0.000000
dtype: float64


In [4]:
# Find rows where all three admission criteria are null
admission_cols = ['covid_admissions', 'influenza_admissions', 'rsv_admissions']
all_null_admissions = df[df[admission_cols].isnull().all(axis=1)]
print(f"Total Invalid Rows: {len(all_null_admissions)}")

# Display a sample of the invalid rows
display(all_null_admissions.head(3))

Total Invalid Rows: 55


,state,week,covid_admissions,influenza_admissions,rsv_admissions,total_respiratory_admissions,aqi_mean,aqi_p90,aqi_max,days_reported,category,defining_parameter
3719,MN,2024-10-05 00:00:00,NaN,NaN,NaN,0.0,37.992958,53.0,84,142,Good,PM2.5
7080,MA,2024-10-05 00:00:00,NaN,NaN,NaN,0.0,33.340659,40.0,47,91,Good,Ozone
7408,VI,2024-10-05 00:00:00,NaN,NaN,NaN,0.0,8.500000,8.9,9,2,Good,PM2.5


In [5]:
# Remove the invalid rows from the dataset
df = df.dropna(subset=admission_cols, how='all')

# Sanity check to make sure rows were remove
all_null_admissions = df[df[admission_cols].isnull().all(axis=1)]
print(f"Total Invalid Rows: {len(all_null_admissions)}")

Total Invalid Rows: 0


In [6]:
# For RSV null values imput data by taking average rsv admissions from same state and same month
df_imputed = df.copy()
df_imputed['rsv_admissions'] = df_imputed.groupby(['state', pd.to_datetime(df_imputed['week']).dt.to_period('M')])['rsv_admissions'].transform(
    lambda x: x.fillna(x.mean())
)

In [7]:
# Only a few null values were filled in meaning that for much of the dataset, in several states there is no RSV admission 
rsv_null_values = df_imputed['rsv_admissions'].isnull().sum()

print(f"Total null rsv_admission rows: {rsv_null_values}")

Total null rsv_admission rows: 8484


In [8]:
# For the rest of the null rsv admissions, NaN will be filled in by zeros
df_filled = df_imputed.fillna({"rsv_admissions": 0})

In [9]:
missing_values = df_filled.isnull().sum()

print("Missingness Values (Total):")
print(missing_values[applicable_variables])

Missingness Values (Total):
covid_admissions        0
influenza_admissions    0
rsv_admissions          0
days_reported           0
category                0
defining_parameter      0
dtype: int64


In [10]:
# Create the DB file!!!
db1_path = PROCESSED_DIR / "milestone1.db"
db2_path = PROCESSED_DIR / "milestone2.db"

# Copy milestone's database and update tables in the copy
shutil.copy(db1_path, db2_path)

conn = _sqlite3.connect(db2_path)

conn.execute("DROP TABLE IF EXISTS merged_m1")

df_filled.to_sql("merged_m2", conn, index=False, if_exists="replace")

cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
print("Tables in new database:")
print(cursor.fetchall())

conn.close()

Tables in new database:
[('cdc_m1',), ('aqi_m1',), ('merged_m2',)]


### ML Model Data Preparation and Feature Engineering

In [11]:
db_path = PROCESSED_DIR / "milestone2.db"
conn = _sqlite3.connect(db_path)

df = pd.read_sql_query("SELECT * FROM merged_m2", conn)

df['era'] = pd.cut(df['week'],
    bins=[pd.Timestamp('2020-01-01'),
          pd.Timestamp('2023-01-01'),
          pd.Timestamp('2026-01-01')],
    labels=['covid', 'post_covid']
)

print("Data distribution by era:")
print(df['era'].value_counts())

Data distribution by era:
era
post_covid    7460
covid         5565
Name: count, dtype: int64


In [25]:
df['baseline'] = df.groupby(['state', 'era'])['total_respiratory_admissions'].transform('median')

baseline_comparison = df.groupby(['state', 'era'])['baseline'].first()
print(baseline_comparison.head(10))

post_covid_baselines = df[df['era'] == 'post_covid'].groupby('state')['total_respiratory_admissions'].median()
df['baseline'] = df['state'].map(post_covid_baselines)

display(df.head(100))

state  era       
AK     covid           57.400
       post_covid      24.285
AL     covid          499.860
       post_covid     177.440
AR     covid          315.280
       post_covid      81.425
AZ     covid          603.240
       post_covid     190.700
CA     covid         3535.880
       post_covid    1199.495
Name: baseline, dtype: float64


/var/folders/f8/bly1gbkj57j3my565938mkkh0000gp/T/ipykernel_2103/3102793144.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df['baseline'] = df.groupby(['state', 'era'])['total_respiratory_admissions'].transform('median')
/var/folders/f8/bly1gbkj57j3my565938mkkh0000gp/T/ipykernel_2103/3102793144.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  baseline_comparison = df.groupby(['state', 'era'])['baseline'].first()


,state,week,covid_admissions,influenza_admissions,rsv_admissions,total_respiratory_admissions,aqi_mean,aqi_p90,aqi_max,days_reported,category,defining_parameter,era,baseline,spike_ratio,is_spike
0,AL,2021-10-09 00:00:00,1230.00,8.50,0.0,1238.50,34.580645,52.6,80,93,Good,Ozone,covid,177.44,3.782950,True
1,AL,2021-10-16 00:00:00,962.57,4.00,0.0,966.57,40.752577,57.0,67,97,Good,PM2.5,covid,177.44,2.952350,True
2,AL,2021-10-23 00:00:00,707.91,8.00,0.0,715.91,40.423913,55.0,65,92,Good,Ozone,covid,177.44,2.186719,True
3,AL,2021-10-30 00:00:00,554.86,8.86,0.0,563.72,30.712766,44.4,70,94,Good,Ozone,covid,177.44,1.721861,True
4,AL,2021-11-06 00:00:00,451.71,10.14,0.0,461.85,38.166667,52.0,57,66,Good,PM2.5,covid,177.44,1.410703,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,AL,2023-08-05 00:00:00,183.03,6.00,0.0,189.03,52.464646,60.2,255,99,Good,PM2.5,post_covid,177.44,0.577385,False
96,AL,2023-08-12 00:00:00,211.29,5.71,0.0,217.00,38.770833,53.0,66,96,Good,PM2.5,post_covid,177.44,0.662818,False
97,AL,2023-08-19 00:00:00,247.02,8.21,0.0,255.23,45.181818,59.0,84,99,Good,PM2.5,post_covid,177.44,0.779590,False
98,AL,2023-08-26 00:00:00,301.29,8.81,0.0,310.10,60.010000,72.2,126,100,Moderate,PM2.5,post_covid,177.44,0.947188,False


In [26]:
# Compute spike ratio to determine if spike occured (Spike is defined as total respiratory admissions increasing by over 1.5 times the baseline)
df['spike_ratio'] = df['total_respiratory_admissions'] / df['baseline']
df['is_spike'] = (df['spike_ratio'] > 1.5)

display(df.head(100))

,state,week,covid_admissions,influenza_admissions,rsv_admissions,total_respiratory_admissions,aqi_mean,aqi_p90,aqi_max,days_reported,category,defining_parameter,era,baseline,spike_ratio,is_spike
0,AL,2021-10-09 00:00:00,1230.00,8.50,0.0,1238.50,34.580645,52.6,80,93,Good,Ozone,covid,177.44,6.979824,True
1,AL,2021-10-16 00:00:00,962.57,4.00,0.0,966.57,40.752577,57.0,67,97,Good,PM2.5,covid,177.44,5.447306,True
2,AL,2021-10-23 00:00:00,707.91,8.00,0.0,715.91,40.423913,55.0,65,92,Good,Ozone,covid,177.44,4.034660,True
3,AL,2021-10-30 00:00:00,554.86,8.86,0.0,563.72,30.712766,44.4,70,94,Good,Ozone,covid,177.44,3.176961,True
4,AL,2021-11-06 00:00:00,451.71,10.14,0.0,461.85,38.166667,52.0,57,66,Good,PM2.5,covid,177.44,2.602852,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,AL,2023-08-05 00:00:00,183.03,6.00,0.0,189.03,52.464646,60.2,255,99,Good,PM2.5,post_covid,177.44,1.065318,False
96,AL,2023-08-12 00:00:00,211.29,5.71,0.0,217.00,38.770833,53.0,66,96,Good,PM2.5,post_covid,177.44,1.222949,False
97,AL,2023-08-19 00:00:00,247.02,8.21,0.0,255.23,45.181818,59.0,84,99,Good,PM2.5,post_covid,177.44,1.438402,False
98,AL,2023-08-26 00:00:00,301.29,8.81,0.0,310.10,60.010000,72.2,126,100,Moderate,PM2.5,post_covid,177.44,1.747633,True


In [27]:
# Update DB File
conn = _sqlite3.connect(db2_path)

df.to_sql("merged_m2", conn, index=False, if_exists="replace")

cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
print("Tables in new database:")
print(cursor.fetchall())

conn.close()

Tables in new database:
[('cdc_m1',), ('aqi_m1',), ('merged_m2',)]
